# Document Similarity and Clustering

In Natural Language Processing (NLP) and Information Retrieval (IR), it is often necessary to quantify how similar two documents are to one another, or to group similar documents together. 

This notebook explores:
1. **Document Representation**: How we convert text into numerical vectors (Bag of Words, TF-IDF).
2. **Similarity Measures**: How we calculate the distance or similarity between these vectors (Cosine, Euclidean, Jaccard).
3. **Document Clustering**: How we group documents into clusters using unsupervised learning (K-Means).
4. **Clustering Evaluation**: How we evaluate the quality of our clusters using internal and external measures.

In [1]:
# Import necessary libraries
import numpy as np
import pandas as pd

# Feature extraction
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# Similarity metrics
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

# Clustering
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances

# Clustering evaluation metrics
from sklearn.metrics import silhouette_score, davies_bouldin_score # Internal measures
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score # External measures

## 1. Document Representation

To analyze text mathematically, we must first convert it into numbers. Here we define a small corpus of text documents and their corresponding ground truth categories.

In [2]:
# Define a sample corpus of documents
documents = [
    "Natural language processing is a fascinating field.",
    "Machine learning brings fascinating concepts to natural language processing.",
    "The quick brown fox jumps over the lazy dog.",
    "Artificial intelligence encompasses machine learning and natural language processing.",
    "A quick dog jumps over the brown fox.",
    "Deep learning and machine learning are subsets of artificial intelligence."
]

# Ground truth labels representing the "topic" of each document
# 0: Technology / AI / NLP
# 1: Animals / Fox / Dog
true_labels = [0, 0, 1, 0, 1, 0]

print("Sample Corpus:")
for i, doc in enumerate(documents):
    print(f"Doc {i}: {doc}")

Sample Corpus:
Doc 0: Natural language processing is a fascinating field.
Doc 1: Machine learning brings fascinating concepts to natural language processing.
Doc 2: The quick brown fox jumps over the lazy dog.
Doc 3: Artificial intelligence encompasses machine learning and natural language processing.
Doc 4: A quick dog jumps over the brown fox.
Doc 5: Deep learning and machine learning are subsets of artificial intelligence.


### 1.1 Bag of Words (CountVectorizer)

The simplest representation is **Bag of Words (BoW)**. It counts the frequency of each word in the document, completely ignoring grammar and word order. Let's create a BoW matrix for our documents.

In [3]:
count_vectorizer = CountVectorizer()
bow_matrix = count_vectorizer.fit_transform(documents)

print("Bag of Words feature names:")
print(count_vectorizer.get_feature_names_out())
print("\nBoW Matrix Shape:", bow_matrix.shape)

Bag of Words feature names:
['and' 'are' 'artificial' 'brings' 'brown' 'concepts' 'deep' 'dog'
 'encompasses' 'fascinating' 'field' 'fox' 'intelligence' 'is' 'jumps'
 'language' 'lazy' 'learning' 'machine' 'natural' 'of' 'over' 'processing'
 'quick' 'subsets' 'the' 'to']

BoW Matrix Shape: (6, 27)


### 1.2 TF-IDF (Term Frequency-Inverse Document Frequency)

**TF-IDF** improves upon BoW by penalizing words that appear frequently across all documents (like "the", "is", "a") and giving more weight to rare, informative words.

In [24]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (6, 27)


In [27]:
tfidf_matrix.data

array([0.34147953, 0.34147953, 0.34147953, 0.49324485, 0.40446783,
       0.49324485, 0.28102627, 0.28102627, 0.28102627, 0.33286354,
       0.28102627, 0.28102627, 0.40592406, 0.40592406, 0.40592406,
       0.59009739, 0.2950487 , 0.2950487 , 0.2950487 , 0.2950487 ,
       0.2950487 , 0.35980921, 0.2950487 , 0.29754518, 0.29754518,
       0.29754518, 0.29754518, 0.29754518, 0.35242947, 0.35242947,
       0.42978454, 0.35242947, 0.37796447, 0.37796447, 0.37796447,
       0.37796447, 0.37796447, 0.37796447, 0.37796447, 0.23867517,
       0.47735035, 0.28270049, 0.28270049, 0.28270049, 0.34475067,
       0.34475067, 0.34475067, 0.34475067])

In [37]:
csr_matrix(tfidf_matrix).toarray()

array([[0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.40446783,
        0.49324485, 0.        , 0.        , 0.49324485, 0.        ,
        0.34147953, 0.        , 0.        , 0.        , 0.34147953,
        0.        , 0.        , 0.34147953, 0.        , 0.        ,
        0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.40592406, 0.        ,
        0.40592406, 0.        , 0.        , 0.        , 0.33286354,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.28102627, 0.        , 0.28102627, 0.28102627, 0.28102627,
        0.        , 0.        , 0.28102627, 0.        , 0.        ,
        0.        , 0.40592406],
       [0.        , 0.        , 0.        , 0.        , 0.2950487 ,
        0.        , 0.        , 0.2950487 , 0.        , 0.        ,
        0.        , 0.2950487 , 0.        , 0.        , 0.2950487 ,
        0.        , 0.35980921, 0.        , 0.    

In [40]:
len(tfidf_vectorizer.get_feature_names_out())

27

---
## 2. Document Similarity Measures

Once documents are vectorized, we can compute how similar they are. We will demonstrate three common methods.

### 2.1 Cosine Similarity
**Definition**: Cosine similarity measures the cosine of the angle between two multi-dimensional vectors. 
**Why use it?**: It is independent of document length (magnitude). Two documents can be very similar in their word distribution even if one is much longer than the other.

In [5]:
# Calculate Cosine Similarity on both TF-IDF and BoW matrices
cos_sim_tfidf = cosine_similarity(tfidf_matrix)
cos_sim_bow = cosine_similarity(bow_matrix)

df_cos_tfidf = pd.DataFrame(cos_sim_tfidf, 
                            index=[f"Doc {i}" for i in range(len(documents))], 
                            columns=[f"Doc {i}" for i in range(len(documents))])

print("=== Cosine Similarity (TF-IDF) ===")
display(df_cos_tfidf.round(3))

=== Cosine Similarity (TF-IDF) ===


,Doc 0,Doc 1,Doc 2,Doc 3,Doc 4,Doc 5
Doc 0,1.000,0.423,0.000,0.305,0.000,0.000
Doc 1,0.423,1.000,0.000,0.418,0.000,0.201
Doc 2,0.000,0.000,1.000,0.000,0.892,0.000
Doc 3,0.305,0.418,0.000,1.000,0.000,0.512
Doc 4,0.000,0.000,0.892,0.000,1.000,0.000
Doc 5,0.000,0.201,0.000,0.512,0.000,1.000


### 2.2 Euclidean Distance

**Definition**: Euclidean distance measures the straight-line distance between two points in Euclidean space.
**Note**: Unlike Cosine Similarity (where higher is more similar), for Euclidean Distance, **lower values mean documents are closer (more similar)**. It is highly sensitive to document length, which makes TF-IDF (which normalizes vectors) a better representation than BoW here.

In [6]:
# Calculate Euclidean Distance
euc_dist_tfidf = euclidean_distances(tfidf_matrix)

df_euc_tfidf = pd.DataFrame(euc_dist_tfidf, 
                            index=[f"Doc {i}" for i in range(len(documents))], 
                            columns=[f"Doc {i}" for i in range(len(documents))])

print("=== Euclidean Distance (TF-IDF) ===")
display(df_euc_tfidf.round(3))

=== Euclidean Distance (TF-IDF) ===


,Doc 0,Doc 1,Doc 2,Doc 3,Doc 4,Doc 5
Doc 0,0.000,1.075,1.414,1.179,1.414,1.414
Doc 1,1.075,0.000,1.414,1.079,1.414,1.264
Doc 2,1.414,1.414,0.000,1.414,0.464,1.414
Doc 3,1.179,1.079,1.414,0.000,1.414,0.988
Doc 4,1.414,1.414,0.464,1.414,0.000,1.414
Doc 5,1.414,1.264,1.414,0.988,1.414,0.000


### 2.3 Jaccard Similarity

**Definition**: Jaccard similarity is the size of the intersection divided by the size of the union of the sample sets. 
**Why use it?**: It is an intuitive measure that evaluates word overlap between sets of words, without relying on vector mathematics. It completely ignores term frequencies.

In [7]:
def jaccard_similarity(doc1, doc2):
    # Convert documents to sets of unique words
    set1 = set(doc1.lower().split())
    set2 = set(doc2.lower().split())
    
    intersection = set1.intersection(set2)
    union = set1.union(set2)
    
    return len(intersection) / len(union)

n = len(documents)
jac_sim = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        jac_sim[i, j] = jaccard_similarity(documents[i], documents[j])

df_jac = pd.DataFrame(jac_sim, 
                      index=[f"Doc {i}" for i in range(len(documents))], 
                      columns=[f"Doc {i}" for i in range(len(documents))])

print("=== Jaccard Similarity ===")
display(df_jac.round(3))

=== Jaccard Similarity ===


,Doc 0,Doc 1,Doc 2,Doc 3,Doc 4,Doc 5
Doc 0,1.000,0.231,0.000,0.143,0.071,0.000
Doc 1,0.231,1.000,0.000,0.385,0.000,0.125
Doc 2,0.000,0.000,1.000,0.000,0.455,0.000
Doc 3,0.143,0.385,0.000,1.000,0.000,0.286
Doc 4,0.071,0.000,0.455,0.000,1.000,0.000
Doc 5,0.000,0.125,0.000,0.286,0.000,1.000


---
## 3. Document Clustering

Now that we can mathematically relate documents, we can group them using **K-Means Clustering**. Based on our corpus, we expect two major clusters: Technology (AI/NLP) and Animals (Fox/Dog).

In [8]:
# We chose K=2 because we engineered the labels that way
num_clusters = 2
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
predicted_labels = kmeans.fit_predict(tfidf_matrix)

print(f"Predicted Labels: {predicted_labels}")
print(f"True Labels:      {true_labels}")

Predicted Labels: [1 1 0 1 0 1]
True Labels:      [0, 0, 1, 0, 1, 0]


---
## 4. Clustering Evaluation Measures

How do we know if our clusters are good? We use evaluation metrics. They are divided into two categories:
1. **Internal Measures**: Evaluate based solely on the data and the resulting clusters (when we have no ground truth).
2. **External Measures**: Evaluate the clusters by comparing them to a known ground truth (like a supervised labels array).

### 4.1 Internal Measures

- **Cohesion (Intra-cluster Similarity)**: Measures how closely related objects in the *same* cluster are. Lower values indicate tighter, more strongly cohesive clusters.
- **Inter-cluster Dissimilarity (Separation)**: Measures how far apart the distinct clusters are from each other. Higher is better.
- **Silhouette Score**: Calculates how similar an object is to its own cluster compared to other clusters. Bounded between -1 and 1. Near 1 is excellent, near 0 indicates overlapping clusters.
- **Davies-Bouldin Index**: Formulated as the average similarity measure of each cluster with its most similar cluster. Lower values indicate better clustering separation.

In [11]:
print("\n--- Internal Measures ---")

# 1. Cohesion (Sum of squared distances of samples to their closest cluster center)
cohesion = kmeans.inertia_
print(f"Cohesion (Inertia): {cohesion:.4f}")


--- Internal Measures ---
Cohesion (Inertia): 2.1786


In [12]:

# 2. Inter-cluster Dissimilarity (Average distance between centroids)
centroids = kmeans.cluster_centers_
if num_clusters > 1:
    # Metric Euclidean is standard for K-Means cluster centers
    inter_cluster_dist = pairwise_distances(centroids, metric='euclidean')
    avg_inter_dist = np.sum(inter_cluster_dist) / (num_clusters * (num_clusters - 1))
    print(f"Inter-cluster Dissimilarity: {avg_inter_dist:.4f}")
else:
    print("Inter-cluster Dissimilarity: N/A (Only 1 cluster)")

Inter-cluster Dissimilarity: 1.1952


In [13]:

# 3. Silhouette Score
sil_score = silhouette_score(tfidf_matrix, predicted_labels)
print(f"Silhouette Score: {sil_score:.4f}")

Silhouette Score: 0.3407


In [14]:

# 4. Davies-Bouldin Index
db_score = davies_bouldin_score(tfidf_matrix.toarray(), predicted_labels)
print(f"Davies-Bouldin Index: {db_score:.4f}")

Davies-Bouldin Index: 0.7927


### 4.2 External Measures

- **Purity**: The most intuitive measure. It assigns each cluster to the class which is most frequent in the cluster, and then calculates the accuracy (correctly assigned items / total items). Ranges from 0 to 1.
- **Adjusted Rand Index (ARI)**: Conceptually, counts the pairs of samples that were grouped together vs. separated in both the true and predicted sets. It is 'Adjusted' to account for chance grouping. Ranges from -1 to 1 (1 is perfect clustering).
- **Normalized Mutual Information (NMI)**: An information-theoretic measure of the mutual dependence between the predicted labels and true labels, normalized against entropy. Ranges from 0 (no mutual info) to 1 (perfect correlation).

In [69]:
print("\n--- External Measures ---")

# 1. Purity
def purity_score(y_true, y_pred):
    # Compute contingency matrix (intersection counts between true and predicted clusters)
    contingency_matrix = pd.crosstab(pd.Series(y_true, name='True'), pd.Series(y_pred, name='Predicted'))
    # Max over columns (most frequent class in each cluster), summed, over total points
    return np.sum(np.amax(contingency_matrix.values, axis=0)) / np.sum(contingency_matrix.values)

purity = purity_score(true_labels, predicted_labels)
print(f"Purity: {purity:.4f}")


--- External Measures ---
Purity: 1.0000


In [16]:

# 2. Adjusted Rand Index (ARI)
ari_score = adjusted_rand_score(true_labels, predicted_labels)
print(f"Adjusted Rand Index (ARI): {ari_score:.4f}")

Adjusted Rand Index (ARI): 1.0000


In [17]:

# 3. Normalized Mutual Information (NMI)
nmi_score = normalized_mutual_info_score(true_labels, predicted_labels)
print(f"Normalized Mutual Information (NMI): {nmi_score:.4f}")

Normalized Mutual Information (NMI): 1.0000
